# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [7]:
!pip install folium pandas
#import piplite
#await piplite.install(['folium'])
#await piplite.install(['pandas'])

In [8]:
import folium
import pandas as pd

In [9]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [12]:
# Download and read the `spacex_launch_geo.csv`
#from js import fetch
#import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
#resp = await fetch(URL)
#spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(URL)
spacex_df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


Now, you can take a look at what are the coordinates for each site.


In [13]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [14]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)
site_map 

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [15]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [16]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label

# Iterate through each launch site in the dataframe
for index, row in launch_sites_df.iterrows():  # launch_sites_df have been already defined with columns like 'Launch Site', 'Lat', 'Long' above
    coordinate = [row['Lat'], row['Long']]
    site_name = row['Launch Site']

  
    # 1. Create and add a Circle object for each launch site with a blue circle and popup
    circle = folium.Circle(
        location=coordinate, 
        radius=1000, 
        color='#d35400', 
        fill=True
    ).add_child(folium.Popup(site_name))
    
    # 2. Create and add a Marker object with a text label
    marker = folium.map.Marker(
        location=coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % site_name,
        )
    )

   
    # Add both to the site map
    site_map.add_child(circle)
    site_map.add_child(marker)

# Display the map
site_map

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [17]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [18]:
from folium.plugins import MarkerCluster

# Initialize the marker cluster
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [19]:

# Apply a function to check the value of `class` column
#Create and Apply the Color Function, #  If class=1, marker_color value will be green, # If class=0, marker_color value will be red
# So, we'll write a quick function to map the class values (1 or 0) to their respective colors and apply it to a new column called marker_color
# Function to assign color based on the launch class
def assign_marker_color(launch_class):
    if launch_class == 1:
        return 'green'
    else:
        return 'red'

# Apply the function to create the new marker_color column
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(5)

,Launch Site,Lat,Long,class,marker_color
51,CCAFS SLC-40,28.563197,-80.57682,0,red
52,CCAFS SLC-40,28.563197,-80.57682,0,red
53,CCAFS SLC-40,28.563197,-80.57682,0,red
54,CCAFS SLC-40,28.563197,-80.57682,1,green
55,CCAFS SLC-40,28.563197,-80.57682,0,red


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [20]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed
for index, record in spacex_df.iterrows():
    # Extract the coordinate for the current launch record (Exactly 4 spaces)
    coordinate = [record['Lat'], record['Long']]
    
    # Create a Marker object with its coordinate and customized color icon
    marker = folium.Marker(
        location=coordinate,
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=folium.Popup(f"Site: {record['Launch Site']}<br>Outcome: {'Success' if record['class'] == 1 else 'Failure'}")
    )
    
    # Add the marker to the marker cluster (Exactly 4 spaces)
    marker_cluster.add_child(marker)

# Display the map (No indentation)
site_map


Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [21]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [22]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1) # Converts latitude 1 from degrees to radians because math functions require radians
    lon1 = radians(lon1) # Converts longitude 1 from degrees to radians
    lat2 = radians(lat2) # Converts latitude 2 from degrees to radians
    lon2 = radians(lon2) # Converts longitude 2 from degrees to radians

    dlon = lon2 - lon1 # Calculates the difference between the two longitudes
    dlat = lat2 - lat1 # Calculates the difference between the two latitudes

    # sin: Calculates the sine of half the latitude/longitude difference
    # cos: Calculates the cosine of the latitudes to account for the Earth's curvature shrinking near the poles
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    
    # sqrt: Computes the square root of the components for the ratio
    # atan2: Computes the arc tangent of y/x, safely returning the correct angle in radians between -π and π
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c  # Multiplis the angular distance (c) by Earth's radius (R) to get the final distance
    return distance
    

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [23]:
# find coordinate of the closet coastline
# e.g.,: Lat: 28.56367  Lon: -80.57163
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

import folium
from folium.plugins import MousePosition

# 1.  Ensure MousePosition is added to your map so you can find coordinates easily
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)
site_map.add_child(mouse_position)

# 2. Define  coordinates (Replace these example values with  actual data)
# Example: Using 'CCAFS SLC-40' launch site coordinates
launch_site_lat = 28.562302
launch_site_lon = -80.577356

# Example: Coordinate of the closest coastline found using MousePosition
coastline_lat = 28.56367
coastline_lon = -80.57163

# 3. Calculate the distance
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
distance_coastline

0.5797120571581802

In [24]:
# find coordinate of from KSC LC-39A to the closet coastline
launch_site2_lat = 28.573255
launch_site2_lon = -80.646895
coastline_lat = 28.58109
coastline_lon  = -80.61355
distance_coastline2 = calculate_distance(launch_site2_lat, launch_site2_lon, coastline_lat, coastline_lon)
distance_coastline2

3.371683766512299

In [25]:
# Create and add a folium.Marker on the selected closest coastline point on the map
# Display the distance between coastline point and launch site using the icon property 

from folium.features import DivIcon

# We already have these variables defined from your previous steps:
           # site_map = folium.Map(...)
           # launch_site_lat, launch_site_lon = ...
           # coastline_lat, coastline_lon = ...
           # distance = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# 1. Define the coordinate for the closest coastline point
coastline_coordinate = [coastline_lat, coastline_lon]

# 2. Create the distance marker using DivIcon
distance_marker = folium.Marker(
    location=coastline_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12px; color:#d35400;"><b>{distance_coastline:.2f} KM</b></div>',
       
    )
)

# 3. Add the marker to the map
site_map.add_child(distance_marker)

# 4.  Create and add a folium.PolyLine between the launch site and the coastline  - Optional
launch_coordinate = [launch_site_lat, launch_site_lon]
lines = folium.PolyLine(
    locations=[launch_coordinate, coastline_coordinate],
    weight=1,
    color='blue'
)
site_map.add_child(lines)

# Display map 
site_map


*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [26]:
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
# Create a list containing both the launch site and coastline coordinates
coordinates = [[launch_site_lat, launch_site_lon], coastline_coordinate]
# Create the folium.PolyLine object
lines = folium.PolyLine(
    locations=coordinates, 
    weight=1,
    color='blue' # Added color to make it clearly visible on the map
)
# Add the polyline to the map
site_map.add_child(lines)

# Dynamically adjust the map zoom to fit both points on the screen
site_map.fit_bounds(coordinates)

# Display map
site_map


Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


### Create a marker with distance to a closest city, railway, highway, etc.
### Draw a line between the marker to the launch site


In [27]:
from folium.features import DivIcon

# 1. Define the target coordinate 
# Example: Closest City, Railway, or Highway
poi_coordinate = [28.46627, -80.5631] 

# 2. Calculate the distance using your previously defined function
# (launch_site_lat and launch_site_lon are already defined)
distance_poi = calculate_distance(launch_site_lat, launch_site_lon, poi_coordinate[0], poi_coordinate[1])


# 3. Create and add the distance text marker at the POI location
poi_distance_marker = folium.Marker(
    location=poi_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400; font-weight: bold;">{:10.2f} KM</div>'.format(distance_poi),
    )
)
site_map.add_child(poi_distance_marker)


# 4. Create a list of coordinates for the line
poi_line_coordinates = [[launch_site_lat, launch_site_lon], poi_coordinate]

# 5. Create and add the folium.PolyLine object
poi_line = folium.PolyLine(
    locations=poi_line_coordinates,
    weight=1,
    color='green' # Using green to distinguish it from the coastline line
)
site_map.add_child(poi_line)


# 6. Adjust map zoom to fit the new line perfectly
site_map.fit_bounds(poi_line_coordinates)

# Display map
site_map

In [28]:
# Additional
# Create a marker with distance of from KSC LC-39A to a closest city
city_lat = 28.52806
city_lon = -80.65615
distance_city2 = calculate_distance(launch_site2_lat, launch_site2_lon, city_lat, city_lon)
coordinate = [city_lat, city_lon]
distance_marker = folium.Marker(
    coordinate,
    icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0),html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_city2),
                ))
site_map.add_child(distance_marker)
# Create a `folium.PolyLine` object using the coastline coordinates and launch site coordinate
lines=folium.PolyLine(locations=[[launch_site2_lat,launch_site2_lon],[city_lat,city_lon]], weight=1)
site_map.add_child(lines)
site_map

After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.
